# 02 - BraTS2020 Data Pipeline

This notebook validates the reusable data pipeline for
BraTS2020 brain tumor segmentation.

- MRI modalities: T1, T1ce, T2, FLAIR
- Input volume: 240 × 240 × 155
- Training patch: 96 × 96 × 96
- Data split seed: 42

In [2]:
# 02 DATA PIPELINE - STEP 1: SETUP REPOSITORY

from pathlib import Path
import subprocess
import sys

REPO_URL = (
    "https://github.com/"
    "tthanh81/brats2020-segmentation.git"
)

BRANCH = "refactor/data-pipeline"

PROJECT_ROOT = Path(
    "/kaggle/working/brats2020-segmentation"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"
)

# Clone repository nếu chưa tồn tại.
if not PROJECT_ROOT.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )

    print("Repository cloned successfully.")

else:
    assert (PROJECT_ROOT / ".git").is_dir(), (
        "PROJECT_ROOT exists but is not a Git repository."
    )

    print("Existing repository detected.")

# Kiểm tra branch đang sử dụng.
current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()

assert current_branch == BRANCH, (
    f"Expected {BRANCH}, found {current_branch}"
)

# Thêm repository vào Python path.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Kiểm tra source code.
required_paths = [
    "src/data/preprocessing.py",
    "src/data/dataset.py",
    "src/data/patches.py",
    "src/models/unet3d.py",
    "splits/split_manifest_seed42.csv",
]

for relative_path in required_paths:

    path = PROJECT_ROOT / relative_path

    assert path.is_file(), (
        f"Missing file: {relative_path}"
    )

# Kiểm tra dataset Kaggle.
assert DATA_ROOT.is_dir(), (
    f"Dataset not found: {DATA_ROOT}. "
    "Check the Kaggle Input panel."
)

print("\nBranch:", current_branch)
print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Required files:", len(required_paths), "verified")

print("\nPASS: Data pipeline environment ready.")

Cloning into '/kaggle/working/brats2020-segmentation'...


Repository cloned successfully.

Branch: refactor/data-pipeline
Project root: /kaggle/working/brats2020-segmentation
Data root: /kaggle/input/datasets/awsaf49/brats20-dataset-training-validation
Required files: 5 verified

PASS: Data pipeline environment ready.


## 2. Dataset Split Manifest

Load the fixed patient-level split generated with random seed 42.

- Train: 257 subjects
- Internal validation: 55 subjects
- Test: 56 subjects

The split is loaded from the repository, not regenerated.

In [3]:
# 02 DATA PIPELINE - STEP 2: LOAD SPLIT MANIFEST

import pandas as pd
from pathlib import Path

MANIFEST_PATH = (
    PROJECT_ROOT / "splits/split_manifest_seed42.csv"
)

assert MANIFEST_PATH.is_file(), (
    f"Manifest not found: {MANIFEST_PATH}"
)

# 1. Đọc manifest cố định.
split_manifest = pd.read_csv(MANIFEST_PATH)

# 2. Kiểm tra các cột bắt buộc.
required_columns = {
    "subject_id",
    "t1",
    "t1ce",
    "t2",
    "flair",
    "seg",
    "split",
}

assert required_columns.issubset(
    split_manifest.columns
), "Manifest is missing required columns."

# 3. Kiểm tra số lượng và tính duy nhất.
assert len(split_manifest) == 368

assert split_manifest["subject_id"].is_unique, (
    "Duplicate subject IDs detected."
)

expected_counts = {
    "train": 257,
    "internal_validation": 55,
    "test": 56,
}

actual_counts = (
    split_manifest["split"]
    .value_counts()
    .to_dict()
)

assert actual_counts == expected_counts, (
    f"Unexpected split counts: {actual_counts}"
)

# 4. Kiểm tra file MRI và segmentation mask.
path_columns = ["t1", "t1ce", "t2", "flair", "seg"]

missing_paths = []

for _, row in split_manifest.iterrows():
    for column in path_columns:

        relative_path = row[column]

        if pd.isna(relative_path):
            missing_paths.append(
                (row["subject_id"], column, "Missing path")
            )
            continue

        full_path = DATA_ROOT / str(relative_path)

        if not full_path.is_file():
            missing_paths.append(
                (row["subject_id"], column, str(full_path))
            )

assert len(missing_paths) == 0, (
    f"Found {len(missing_paths)} missing files. "
    f"Examples: {missing_paths[:5]}"
)

# 5. Khôi phục ba DataFrame.
train_df = (
    split_manifest[split_manifest["split"] == "train"]
    .reset_index(drop=True)
    .copy()
)

val_df = (
    split_manifest[
        split_manifest["split"] == "internal_validation"
    ]
    .reset_index(drop=True)
    .copy()
)

test_df = (
    split_manifest[split_manifest["split"] == "test"]
    .reset_index(drop=True)
    .copy()
)

print("Manifest:", MANIFEST_PATH)

print("\nDataset splits:")
print("Train:", len(train_df))
print("Internal validation:", len(val_df))
print("Test:", len(test_df))

print("\nTotal referenced MRI / segmentation files:",
      len(split_manifest) * len(path_columns))

print("Missing files:", len(missing_paths))

print("\nExample subjects:")
print("Train:", train_df.iloc[0]["subject_id"])
print("Validation:", val_df.iloc[0]["subject_id"])
print("Test:", test_df.iloc[0]["subject_id"])

print("\nPASS: Split manifest loaded and verified.")

Manifest: /kaggle/working/brats2020-segmentation/splits/split_manifest_seed42.csv

Dataset splits:
Train: 257
Internal validation: 55
Test: 56

Total referenced MRI / segmentation files: 1840
Missing files: 0

Example subjects:
Train: BraTS20_Training_001
Validation: BraTS20_Training_002
Test: BraTS20_Training_007

PASS: Split manifest loaded and verified.


## 3. Volume Dataset

Load the BraTSVolumeDataset class from the repository.

Validate:
- Train, validation and test dataset sizes
- MRI channel order and tensor shapes
- Normalized image intensities
- Remapped segmentation labels

In [4]:
# 02 DATA PIPELINE - STEP 3: VOLUME DATASET

import torch

from src.data.preprocessing import MODALITIES
from src.data.dataset import BraTSVolumeDataset

# 1. Khởi tạo ba Dataset từ split manifest.
train_dataset = BraTSVolumeDataset(
    manifest=train_df,
    data_root=DATA_ROOT,
)

val_dataset = BraTSVolumeDataset(
    manifest=val_df,
    data_root=DATA_ROOT,
)

test_dataset = BraTSVolumeDataset(
    manifest=test_df,
    data_root=DATA_ROOT,
)

# 2. Kiểm tra số lượng bệnh nhân.
assert len(train_dataset) == 257
assert len(val_dataset) == 55
assert len(test_dataset) == 56

print("Dataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nMRI modalities:", MODALITIES)

assert MODALITIES == ["t1", "t1ce", "t2", "flair"]

# 3. Đọc một bệnh nhân thực tế.
sample = train_dataset[0]

image = sample["image"]
label = sample["label"]

print("\nSample subject:", sample["subject_id"])
print("Image shape:", tuple(image.shape))
print("Label shape:", tuple(label.shape))
print("Image dtype:", image.dtype)
print("Label dtype:", label.dtype)

# 4. Kiểm tra tensor.
assert image.shape == (4, 240, 240, 155)
assert label.shape == (240, 240, 155)

assert image.dtype == torch.float32
assert label.dtype == torch.uint8

assert torch.isfinite(image).all()

# 5. Kiểm tra segmentation mask.
unique_labels = torch.unique(label).tolist()

print("\nUnique segmentation labels:", unique_labels)

assert set(unique_labels).issubset({0, 1, 2, 3})

# 6. Kiểm tra chuẩn hóa MRI.
print("\nMRI normalization statistics:")

for i, modality in enumerate(MODALITIES):

    channel = image[i]
    foreground = channel[channel != 0]

    assert foreground.numel() > 0

    mean = foreground.mean().item()
    std = foreground.std(unbiased=False).item()

    print(
        f"{modality:5s} | "
        f"mean = {mean:.5f} | "
        f"std = {std:.5f}"
    )

    assert abs(mean) < 0.01
    assert abs(std - 1.0) < 0.01

print("\nPASS: Volume datasets initialized and verified.")

Dataset sizes:
Train: 257
Validation: 55
Test: 56

MRI modalities: ['t1', 't1ce', 't2', 'flair']

Sample subject: BraTS20_Training_001
Image shape: (4, 240, 240, 155)
Label shape: (240, 240, 155)
Image dtype: torch.float32
Label dtype: torch.uint8

Unique segmentation labels: [0, 1, 2, 3]

MRI normalization statistics:
t1    | mean = 0.00000 | std = 1.00000
t1ce  | mean = 0.00000 | std = 1.00000
t2    | mean = -0.00000 | std = 1.00000
flair | mean = -0.00000 | std = 1.00000

PASS: Volume datasets initialized and verified.


## 4. Random 3D Patch Sampling

Validate the patch sampler imported from `src/data/patches.py`.

- Patch size: 96 × 96 × 96
- Positive-center probability: 0.5
- Verify image and label shapes
- Verify crop coordinates and segmentation labels

In [5]:
# 02 DATA PIPELINE - STEP 4: RANDOM 3D PATCH SAMPLING

import torch

from src.data.patches import RandomPatchDataset

PATCH_SIZE = (96, 96, 96)
POS_PROB = 0.5

# 1. Khởi tạo patch dataset từ train_dataset.
train_patch_dataset = RandomPatchDataset(
    base_dataset=train_dataset,
    patch_size=PATCH_SIZE,
    pos_prob=POS_PROB,
)

assert len(train_patch_dataset) == 257

# 2. Lấy một patch thực tế.
torch.manual_seed(42)

patch = train_patch_dataset[0]

image_patch = patch["image"]
label_patch = patch["label"]

print("Subject:", patch["subject_id"])
print("Center type:", patch["center_type"])
print("Crop start:", patch["crop_start"])

print("\nPatch shapes:")
print("Image:", tuple(image_patch.shape))
print("Label:", tuple(label_patch.shape))

# 3. Kiểm tra kích thước và kiểu dữ liệu.
assert image_patch.shape == (4, 96, 96, 96)
assert label_patch.shape == (96, 96, 96)

assert image_patch.dtype == torch.float32
assert label_patch.dtype == torch.uint8

assert torch.isfinite(image_patch).all()

# 4. Kiểm tra nhãn.
unique_labels = torch.unique(label_patch).tolist()

assert set(unique_labels).issubset({0, 1, 2, 3})

print("\nUnique labels:", unique_labels)

# 5. Kiểm tra tọa độ crop.
crop_start = patch["crop_start"]

assert len(crop_start) == 3

for start, patch_dim, volume_dim in zip(
    crop_start,
    PATCH_SIZE,
    (240, 240, 155),
):
    assert 0 <= start <= volume_dim - patch_dim

# 6. Kiểm tra thông tin sampling.
assert patch["center_type"] in {"positive", "random"}

foreground_voxels = int((label_patch > 0).sum().item())
total_voxels = label_patch.numel()

foreground_ratio = (
    foreground_voxels / total_voxels * 100
)

if patch["center_type"] == "positive":
    assert foreground_voxels > 0

print("\nTumor statistics:")
print("Foreground voxels:", foreground_voxels)
print(f"Foreground ratio: {foreground_ratio:.4f}%")

print("\nPASS: Random 3D patch sampling verified.")

Subject: BraTS20_Training_001
Center type: random
Crop start: (144, 28, 0)

Patch shapes:
Image: (4, 96, 96, 96)
Label: (96, 96, 96)

Unique labels: [0]

Tumor statistics:
Foreground voxels: 0
Foreground ratio: 0.0000%

PASS: Random 3D patch sampling verified.


## 5. DataLoader Validation

Create a PyTorch DataLoader for the training patch dataset.

Configuration:
- Batch size: 1
- Shuffle: True
- Patch size: 96 × 96 × 96
- Positive-center probability: 0.5

Verify batch shapes, tensor types, segmentation labels and data integrity.

In [6]:
# 02 DATA PIPELINE - STEP 5: DATALOADER VALIDATION

import torch
from torch.utils.data import DataLoader

BATCH_SIZE = 1

# 1. Tạo DataLoader.
train_patch_loader = DataLoader(
    dataset=train_patch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

print("Number of training subjects:", len(train_patch_dataset))
print("Number of batches:", len(train_patch_loader))
print("Batch size:", BATCH_SIZE)

assert len(train_patch_loader) == 257

# 2. Lấy một batch thực tế.
batch = next(iter(train_patch_loader))

images = batch["image"]
labels = batch["label"]

# 3. Kiểm tra shape.
print("\nBatch shapes:")
print("Images:", tuple(images.shape))
print("Labels:", tuple(labels.shape))

assert images.shape == (1, 4, 96, 96, 96)
assert labels.shape == (1, 96, 96, 96)

# 4. Kiểm tra kiểu dữ liệu.
print("\nTensor dtypes:")
print("Images:", images.dtype)
print("Labels:", labels.dtype)

assert images.dtype == torch.float32
assert labels.dtype == torch.uint8

# 5. Kiểm tra giá trị.
assert torch.isfinite(images).all()

unique_labels = torch.unique(labels).tolist()

assert set(unique_labels).issubset({0, 1, 2, 3})

print("\nBatch information:")
print("Subject:", batch["subject_id"][0])
print("Center type:", batch["center_type"][0])
print("Unique labels:", unique_labels)

foreground_voxels = int((labels > 0).sum().item())
foreground_ratio = foreground_voxels / labels.numel() * 100

print("Foreground voxels:", foreground_voxels)
print(f"Foreground ratio: {foreground_ratio:.4f}%")

# 6. Kiểm tra batch có thể chuẩn bị cho DiceCELoss.
model_targets = labels.unsqueeze(1).long()

assert model_targets.shape == (1, 1, 96, 96, 96)
assert model_targets.dtype == torch.int64

print("\nModel target shape:", tuple(model_targets.shape))

print("\nPASS: Training DataLoader verified.")

Number of training subjects: 257
Number of batches: 257
Batch size: 1

Batch shapes:
Images: (1, 4, 96, 96, 96)
Labels: (1, 96, 96, 96)

Tensor dtypes:
Images: torch.float32
Labels: torch.uint8

Batch information:
Subject: BraTS20_Training_128
Center type: random
Unique labels: [0]
Foreground voxels: 0
Foreground ratio: 0.0000%

Model target shape: (1, 1, 96, 96, 96)

PASS: Training DataLoader verified.


## 6. Sampling Strategy and Loading Performance

Validate both sampling strategies:

- Positive-centered sampling (`pos_prob=1.0`)
- Random-centered sampling (`pos_prob=0.0`)

Measure:
- Presence of tumor voxels
- Foreground ratio
- Time required to load and crop each patch

These are small sanity checks, not full-dataset benchmarks.

In [7]:
# 02 DATA PIPELINE - STEP 6: SAMPLING & TIMING TEST

import time
import torch

from src.data.patches import RandomPatchDataset

PATCH_SIZE = (96, 96, 96)
N_SAMPLES = 3

# 1. Tạo hai sampler kiểm thử.
positive_dataset = RandomPatchDataset(
    base_dataset=train_dataset,
    patch_size=PATCH_SIZE,
    pos_prob=1.0,
)

random_dataset = RandomPatchDataset(
    base_dataset=train_dataset,
    patch_size=PATCH_SIZE,
    pos_prob=0.0,
)

results = []

# Giữ trạng thái random hiện tại.
rng_state = torch.get_rng_state()

try:
    torch.manual_seed(2026)

    for mode, dataset in [
        ("positive", positive_dataset),
        ("random", random_dataset),
    ]:

        print(f"\nTesting {mode} sampling")
        print("-" * 60)

        for i in range(N_SAMPLES):

            start = time.perf_counter()

            # Dùng cùng một bệnh nhân để so sánh.
            patch = dataset[0]

            elapsed = time.perf_counter() - start

            image = patch["image"]
            label = patch["label"]

            tumor_voxels = int(
                (label > 0).sum().item()
            )

            foreground_ratio = (
                tumor_voxels / label.numel() * 100
            )

            # Kiểm tra shape.
            assert image.shape == (4, 96, 96, 96)
            assert label.shape == (96, 96, 96)

            # Kiểm tra sampling.
            assert patch["center_type"] == mode

            if mode == "positive":
                assert tumor_voxels > 0, (
                    "Positive-centered patch has no tumor!"
                )

            results.append({
                "mode": mode,
                "time": elapsed,
                "tumor_voxels": tumor_voxels,
                "foreground_ratio": foreground_ratio,
            })

            print(
                f"Sample {i+1}: "
                f"tumor_voxels={tumor_voxels:,} | "
                f"foreground={foreground_ratio:.4f}% | "
                f"time={elapsed:.2f}s"
            )

finally:
    torch.set_rng_state(rng_state)

# 2. Tổng hợp kết quả.
print("\n" + "=" * 60)
print("SAMPLING SUMMARY")
print("=" * 60)

for mode in ["positive", "random"]:

    subset = [
        result for result in results
        if result["mode"] == mode
    ]

    tumor_patches = sum(
        result["tumor_voxels"] > 0
        for result in subset
    )

    avg_time = sum(
        result["time"]
        for result in subset
    ) / len(subset)

    avg_foreground = sum(
        result["foreground_ratio"]
        for result in subset
    ) / len(subset)

    print(f"\nMode: {mode}")
    print(f"Tumor-containing patches: {tumor_patches}/{len(subset)}")
    print(f"Average foreground ratio: {avg_foreground:.4f}%")
    print(f"Average loading time: {avg_time:.2f}s")

print("\nPASS: Sampling and loading performance verified.")


Testing positive sampling
------------------------------------------------------------
Sample 1: tumor_voxels=182,412 | foreground=20.6177% | time=1.45s
Sample 2: tumor_voxels=156,206 | foreground=17.6557% | time=1.43s
Sample 3: tumor_voxels=152,779 | foreground=17.2683% | time=1.41s

Testing random sampling
------------------------------------------------------------
Sample 1: tumor_voxels=38,030 | foreground=4.2985% | time=1.43s
Sample 2: tumor_voxels=33,467 | foreground=3.7827% | time=1.42s
Sample 3: tumor_voxels=0 | foreground=0.0000% | time=1.46s

SAMPLING SUMMARY

Mode: positive
Tumor-containing patches: 3/3
Average foreground ratio: 18.5139%
Average loading time: 1.43s

Mode: random
Tumor-containing patches: 2/3
Average foreground ratio: 2.6937%
Average loading time: 1.43s

PASS: Sampling and loading performance verified.
